# Week 05: PHASE 2 — Newton's Laws (II) — Forces Predict Motion

*Physics I (PHY101) | 3 Hours | Dr. Arif Solmaz*

## Learning Objectives

By the end of this session you will be able to:

1. Derive and apply the centripetal acceleration formula $a_c = v^2/r$
2. Analyze forces on an object moving in a horizontal or vertical circle
3. Solve banked-curve problems (with and without friction)
4. Determine the minimum speed at the top of a vertical loop (loop-the-loop)
5. Analyze a conical pendulum and relate the cone angle to angular speed
6. Build intuition through interactive simulations of circular-motion scenarios

## Core Mastery Connection

**PHASE 2 — "Forces Predict Motion":** Same F = ma, circular geometry. The net force points inward — predict speeds and limits on curved paths. This week extends Newton's second law to objects moving in circles. The free body diagram still drives everything, but now you apply Sigma F = ma in the radial direction. You will predict the maximum safe speed on a banked curve, the minimum speed to complete a loop, and the angle of a conical pendulum — all from the same core workflow.

---
## 1. Setup

Run the cell below to import everything we need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (8, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print('All imports successful.')

---
## 2. Theory: Uniform Circular Motion

### 2.1 Why does an object moving in a circle accelerate?

Even if the *speed* $v$ is constant, the *velocity* (a vector) keeps changing direction. That change requires an acceleration directed toward the centre of the circle.

| Quantity | Symbol | Formula | SI Unit |
|---|---|---|---|
| Period | $T$ | $T = 2\pi r / v$ | s |
| Frequency | $f$ | $f = 1/T$ | Hz |
| Angular speed | $\omega$ | $\omega = 2\pi f = v/r$ | rad/s |
| Centripetal accel. | $a_c$ | $a_c = v^2/r = \omega^2 r$ | m/s$^2$ |
| Centripetal force | $F_c$ | $F_c = m a_c = mv^2/r$ | N |

### Key idea

> **Centripetal force is not a new type of force.** It is the *net radial force* supplied by tension, gravity, normal force, friction, or any combination that points toward the centre.

### Analogy -- The hammer throw

Think of an athlete spinning a hammer: the cable tension provides the centripetal force. The moment the cable is released the ball flies off in a straight line (Newton's first law).

---
## 3. Interactive Demo 1 -- Circular Motion with Vectors

Watch an object travel around a circle. The **blue arrow** is the velocity (tangent) and the **red arrow** is the centripetal acceleration (toward centre). Use the slider to change the speed.

In [ ]:
def circular_motion_animation(speed=2.0):
    """Animate an object in uniform circular motion showing v and a vectors."""
    R = 3.0  # radius
    omega = speed / R
    T = 2 * np.pi / omega if omega != 0 else 1e6
    dt = 0.05
    n_frames = int(min(T / dt, 200))

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.set_title(f'Uniform Circular Motion  (v = {speed:.1f} m/s, R = {R:.1f} m)', fontsize=13)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')

    # Draw circle path
    theta_path = np.linspace(0, 2*np.pi, 200)
    ax.plot(R*np.cos(theta_path), R*np.sin(theta_path), 'k--', alpha=0.3, lw=1)
    ax.plot(0, 0, 'k+', markersize=10)  # centre

    ball, = ax.plot([], [], 'ko', markersize=12)
    v_arrow = ax.annotate('', xy=(0,0), xytext=(0,0),
                          arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    a_arrow = ax.annotate('', xy=(0,0), xytext=(0,0),
                          arrowprops=dict(arrowstyle='->', color='red', lw=2))
    v_label = ax.text(0, 0, '', color='blue', fontsize=11, fontweight='bold')
    a_label = ax.text(0, 0, '', color='red', fontsize=11, fontweight='bold')
    info_text = ax.text(-4.8, 4.5, '', fontsize=10, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    def init():
        ball.set_data([], [])
        return ball,

    def update(frame):
        t = frame * dt
        theta = omega * t
        x = R * np.cos(theta)
        y = R * np.sin(theta)
        ball.set_data([x], [y])

        # Velocity: tangent direction
        vx = -speed * np.sin(theta)
        vy =  speed * np.cos(theta)
        scale_v = 0.8
        v_arrow.xy = (x + scale_v*vx, y + scale_v*vy)
        v_arrow.set_position((x, y))
        v_label.set_position((x + scale_v*vx*0.5 - 0.6, y + scale_v*vy*0.5 + 0.2))
        v_label.set_text(f'v={speed:.1f}')

        # Acceleration: toward centre
        ac = speed**2 / R
        ax_dir = -np.cos(theta)  # unit vector toward centre
        ay_dir = -np.sin(theta)
        scale_a = 0.3
        a_arrow.xy = (x + scale_a*ac*ax_dir, y + scale_a*ac*ay_dir)
        a_arrow.set_position((x, y))
        a_label.set_position((x + scale_a*ac*ax_dir*0.5 + 0.2, y + scale_a*ac*ay_dir*0.5 - 0.3))
        a_label.set_text(f'$a_c$={ac:.1f}')

        info_text.set_text(
            f'$\omega$ = {omega:.2f} rad/s\n'
            f'T = {T:.2f} s\n'
            f'$a_c$ = v$^2$/R = {ac:.2f} m/s$^2$'
        )
        return ball,

    anim = FuncAnimation(fig, update, init_func=init,
                         frames=n_frames, interval=50, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

interact(circular_motion_animation,
         speed=FloatSlider(min=0.5, max=6.0, step=0.5, value=2.0,
                           description='Speed (m/s)'));

---
## 4. Theory: Banked Curves

A banked curve allows a vehicle to turn without relying entirely on friction.

### Frictionless banked curve

On a frictionless banked road tilted at angle $\theta$ with radius $r$, the normal force provides the centripetal component:

$$\tan\theta = \frac{v^2}{rg}$$

So the *ideal* speed for a given bank angle is:

$$v_{\text{ideal}} = \sqrt{rg\tan\theta}$$

### With friction

| Condition | Friction direction | Speed range |
|---|---|---|
| $v < v_{\text{ideal}}$ | Up the incline (prevents sliding inward) | Lower bound |
| $v = v_{\text{ideal}}$ | Zero friction needed | Ideal |
| $v > v_{\text{ideal}}$ | Down the incline (prevents sliding outward) | Upper bound |

---
## 5. Interactive Demo 2 -- Banked Curve Calculator

Adjust the bank angle, curve radius, and vehicle speed. The diagram shows the forces and tells you whether the vehicle will slide, stay, or need friction.

In [ ]:
def banked_curve(theta_deg=20.0, radius=50.0, speed=15.0, mu=0.3):
    """Banked curve force diagram and analysis."""
    g = 9.81
    theta = np.radians(theta_deg)
    v_ideal = np.sqrt(radius * g * np.tan(theta))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- Left: force diagram ---
    ax1.set_xlim(-3, 3)
    ax1.set_ylim(-2, 3)
    ax1.set_aspect('equal')
    ax1.set_title('Force Diagram (rear view)', fontsize=13)
    ax1.grid(False)

    # Draw banked surface
    L = 2.5
    x_road = [-L*np.cos(theta), L*np.cos(theta)]
    y_road = [L*np.sin(theta), -L*np.sin(theta)]
    ax1.plot(x_road, y_road, 'k-', lw=4)

    # Car position (on the road)
    cx, cy = 0.0, 0.0
    car = plt.Rectangle((cx-0.3, cy-0.15), 0.6, 0.3, angle=np.degrees(-theta),
                         rotation_point='center', fc='steelblue', ec='black', lw=1.5)
    ax1.add_patch(car)

    # Forces (scaled for visibility)
    scale = 0.15
    m = 1.0  # unit mass for diagram
    W = m * g
    N = m * g / np.cos(theta)  # approximate

    # Weight (down)
    ax1.annotate('', xy=(cx, cy - scale*W), xytext=(cx, cy),
                 arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
    ax1.text(cx + 0.15, cy - scale*W/2, 'mg', color='green', fontsize=12, fontweight='bold')

    # Normal (perpendicular to surface, up-left)
    nx = -np.sin(theta)
    ny = np.cos(theta)
    ax1.annotate('', xy=(cx + scale*N*nx, cy + scale*N*ny), xytext=(cx, cy),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
    ax1.text(cx + scale*N*nx - 0.4, cy + scale*N*ny + 0.1, 'N', color='red', fontsize=12, fontweight='bold')

    # Centripetal direction arrow
    ax1.annotate('', xy=(cx - 1.5, cy), xytext=(cx - 0.5, cy),
                 arrowprops=dict(arrowstyle='->', color='purple', lw=2, linestyle='--'))
    ax1.text(cx - 2.0, cy + 0.2, '$a_c$ (to centre)', color='purple', fontsize=10)

    ax1.text(-2.8, 2.6, f'Bank angle: {theta_deg:.0f} deg', fontsize=11)
    ax1.axis('off')

    # --- Right: speed analysis ---
    v_min = np.sqrt(radius * g * (np.tan(theta) - mu) / (1 + mu * np.tan(theta))) if (1 + mu*np.tan(theta)) > 0 and (np.tan(theta) - mu) > 0 else 0
    denom = (1 - mu * np.tan(theta))
    v_max = np.sqrt(radius * g * (np.tan(theta) + mu) / max(denom, 0.01)) if denom > 0 else 999

    ax2.set_xlim(0, max(v_max*1.3, speed*1.3, 40))
    ax2.set_ylim(-1, 1)
    ax2.set_title('Speed Analysis', fontsize=13)
    ax2.set_xlabel('Speed (m/s)')
    ax2.get_yaxis().set_visible(False)

    # Safe zone
    ax2.axvspan(v_min, v_max, alpha=0.2, color='green', label=f'Safe range')
    ax2.axvline(v_ideal, color='blue', ls='--', lw=2, label=f'Ideal = {v_ideal:.1f} m/s')
    ax2.axvline(speed, color='red', ls='-', lw=3, label=f'Your speed = {speed:.1f} m/s')
    if v_min > 0:
        ax2.axvline(v_min, color='orange', ls=':', lw=1.5, label=f'Min = {v_min:.1f} m/s')
    ax2.axvline(v_max, color='orange', ls=':', lw=1.5, label=f'Max = {v_max:.1f} m/s')

    if speed < v_min:
        status = 'SLIDES INWARD (too slow)'
        color = 'red'
    elif speed > v_max:
        status = 'SLIDES OUTWARD (too fast)'
        color = 'red'
    else:
        status = 'SAFE'
        color = 'green'

    ax2.text(0.5, 0.7, status, transform=ax2.transAxes, fontsize=16,
             fontweight='bold', color=color, ha='center',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))
    ax2.legend(loc='lower right', fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f'Ideal (no-friction) speed: {v_ideal:.2f} m/s')
    print(f'With friction (mu={mu}): safe range [{v_min:.1f}, {v_max:.1f}] m/s')
    print(f'Centripetal accel needed: a_c = v^2/r = {speed**2/radius:.2f} m/s^2')

interact(banked_curve,
         theta_deg=FloatSlider(min=5, max=60, step=1, value=20, description='Angle (deg)'),
         radius=FloatSlider(min=10, max=200, step=5, value=50, description='Radius (m)'),
         speed=FloatSlider(min=1, max=50, step=0.5, value=15, description='Speed (m/s)'),
         mu=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.3, description='Friction'));

---
## 6. Theory: Loop-the-Loop

At the **top** of a vertical loop of radius $R$, both gravity and the normal force point downward (toward the centre):

$$mg + N = \frac{mv_{\text{top}}^2}{R}$$

The minimum speed occurs when $N = 0$ (object barely maintains contact):

$$v_{\text{min,top}} = \sqrt{gR}$$

Using energy conservation from the bottom to the top of the loop ($h = 2R$):

$$v_{\text{min,bottom}} = \sqrt{5gR}$$

---
## 7. Interactive Demo 3 -- Loop-the-Loop Simulator

Set the entry speed at the bottom of the loop and watch the ball travel around. If the speed is too low it will fall off!

In [ ]:
def loop_the_loop(entry_speed=8.0):
    """Animate a ball going through a vertical loop."""
    g = 9.81
    R = 2.0  # loop radius
    v_min = np.sqrt(5 * g * R)

    # Use energy conservation to get speed at each angle
    # Bottom of loop is at y=0, centre at y=R
    # At angle theta from bottom: y = R - R*cos(theta), x = R*sin(theta)
    n_frames = 120

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-4, 4)
    ax.set_ylim(-1, 5.5)
    ax.set_aspect('equal')
    ax.set_title(f'Loop-the-Loop (R={R} m)', fontsize=13)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')

    # Draw the loop
    theta_loop = np.linspace(0, 2*np.pi, 200)
    ax.plot(R*np.sin(theta_loop), R - R*np.cos(theta_loop), 'k-', lw=2)

    # Approach track
    ax.plot([-4, 0], [0, 0], 'k-', lw=2)
    ax.plot([0, 4], [0, 0], 'k-', lw=2)

    ball, = ax.plot([], [], 'ro', markersize=14)
    trail, = ax.plot([], [], 'r-', alpha=0.3, lw=1)
    info = ax.text(-3.8, 5.0, '', fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    trail_x, trail_y = [], []

    # Precompute trajectory
    positions = []
    # Phase 1: approach from left
    n_approach = 20
    for i in range(n_approach):
        frac = i / n_approach
        positions.append((-4 + 4*frac, 0, entry_speed, 'approach'))

    # Phase 2: loop
    n_loop = 80
    fell_off = False
    fell_angle = None
    for i in range(n_loop):
        theta = 2 * np.pi * i / n_loop
        h = R - R * np.cos(theta)  # height above bottom
        v2 = entry_speed**2 - 2*g*h
        if v2 <= 0:
            fell_off = True
            fell_angle = theta
            break
        v = np.sqrt(v2)
        # Check normal force at this point
        # N = m*v^2/R - mg*cos(theta) (measuring theta from bottom)
        N_over_m = v2/R - g*np.cos(theta)
        if theta > np.pi/2 and theta < 3*np.pi/2 and N_over_m < 0:
            fell_off = True
            fell_angle = theta
            break
        x = R * np.sin(theta)
        y = R - R * np.cos(theta)
        positions.append((x, y, v, 'loop'))

    if fell_off and fell_angle is not None:
        # Projectile from detach point
        x0 = R * np.sin(fell_angle)
        y0 = R - R * np.cos(fell_angle)
        v_at_fall = np.sqrt(max(entry_speed**2 - 2*g*y0, 0.1))
        vx0 = v_at_fall * np.cos(fell_angle)
        vy0 = v_at_fall * np.sin(fell_angle)
        for i in range(30):
            t = i * 0.05
            x = x0 + vx0 * t
            y = y0 + vy0 * t - 0.5*g*t**2
            if y < -0.5:
                break
            positions.append((x, y, 0, 'fall'))
    else:
        # Exit to right
        n_exit = 20
        for i in range(n_exit):
            frac = i / n_exit
            positions.append((4*frac, 0, entry_speed, 'exit'))

    def init():
        ball.set_data([], [])
        trail.set_data([], [])
        trail_x.clear()
        trail_y.clear()
        return ball, trail

    def update(frame):
        if frame < len(positions):
            x, y, v, phase = positions[frame]
            ball.set_data([x], [y])
            trail_x.append(x)
            trail_y.append(y)
            trail.set_data(trail_x, trail_y)

            status = 'COMPLETES LOOP' if not fell_off else 'FALLS OFF'
            scolor = 'green' if not fell_off else 'red'
            info.set_text(
                f'Entry speed: {entry_speed:.1f} m/s\n'
                f'Min needed: {v_min:.1f} m/s\n'
                f'Phase: {phase}\n'
                f'Status: {status}'
            )
            info.set_color(scolor)
        return ball, trail

    anim = FuncAnimation(fig, update, init_func=init,
                         frames=len(positions), interval=50, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

interact(loop_the_loop,
         entry_speed=FloatSlider(min=2, max=15, step=0.5, value=8.0,
                                 description='v_entry (m/s)'));

---
## 8. Theory: Conical Pendulum

A mass $m$ on a string of length $L$ sweeps out a horizontal circle while the string makes angle $\theta$ with the vertical.

**Free-body analysis:**
- Vertical: $T\cos\theta = mg$
- Horizontal (centripetal): $T\sin\theta = m\omega^2 r$

where $r = L\sin\theta$. Dividing:

$$\tan\theta = \frac{\omega^2 r}{g} = \frac{\omega^2 L\sin\theta}{g}$$

$$\cos\theta = \frac{g}{\omega^2 L}$$

$$\omega = \sqrt{\frac{g}{L\cos\theta}}$$

As $\omega$ increases, $\theta$ increases (the pendulum swings wider).

---
## 9. Interactive Demo 4 -- Conical Pendulum Simulator

Change the angular speed and string length to see how the cone angle responds.

In [ ]:
def conical_pendulum(omega=3.0, L=1.5):
    """Animate a conical pendulum in 3D-like top and side views."""
    g = 9.81
    cos_theta = g / (omega**2 * L)

    if cos_theta >= 1.0:
        print(f'Angular speed too low: omega must be > {np.sqrt(g/L):.2f} rad/s for this L.')
        print('The pendulum just hangs vertically.')
        return
    if cos_theta <= 0:
        print('Angular speed too high for this string length.')
        return

    theta = np.arccos(cos_theta)
    r = L * np.sin(theta)
    h = L * np.cos(theta)
    T_tension = g / cos_theta  # T/m
    period = 2 * np.pi / omega

    n_frames = 80
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

    # Side view
    ax1.set_xlim(-2, 2)
    ax1.set_ylim(-2, 0.5)
    ax1.set_aspect('equal')
    ax1.set_title('Side View', fontsize=13)
    ax1.set_xlabel('x (m)')
    ax1.set_ylabel('y (m)')
    ax1.plot(0, 0, 'ks', markersize=10)  # pivot

    # Draw circular path (dashed)
    circle_x = np.linspace(-r, r, 100)
    ax1.plot(circle_x, -h * np.ones_like(circle_x), 'b--', alpha=0.2)

    string_line, = ax1.plot([], [], 'k-', lw=2)
    ball1, = ax1.plot([], [], 'ro', markersize=14)
    info1 = ax1.text(-1.9, 0.3, '', fontsize=9,
                     bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    # Top view
    ax2.set_xlim(-2, 2)
    ax2.set_ylim(-2, 2)
    ax2.set_aspect('equal')
    ax2.set_title('Top View', fontsize=13)
    ax2.set_xlabel('x (m)')
    ax2.set_ylabel('z (m)')
    circ = plt.Circle((0, 0), r, fill=False, ls='--', color='gray', alpha=0.5)
    ax2.add_patch(circ)
    ax2.plot(0, 0, 'ks', markersize=8)
    ball2, = ax2.plot([], [], 'ro', markersize=14)
    radius_line, = ax2.plot([], [], 'k--', alpha=0.3)

    def init():
        return ball1, string_line, ball2, radius_line

    def update(frame):
        phi = 2 * np.pi * frame / n_frames
        bx = r * np.cos(phi)
        bz = r * np.sin(phi)
        by = -h

        # Side view (project onto x-y plane)
        string_line.set_data([0, bx], [0, by])
        ball1.set_data([bx], [by])

        info1.set_text(
            f'theta = {np.degrees(theta):.1f} deg\n'
            f'r = {r:.2f} m\n'
            f'omega = {omega:.1f} rad/s\n'
            f'T = {period:.2f} s\n'
            f'Tension/m = {T_tension:.1f} N/kg'
        )

        # Top view
        ball2.set_data([bx], [bz])
        radius_line.set_data([0, bx], [0, bz])

        return ball1, string_line, ball2, radius_line

    anim = FuncAnimation(fig, update, init_func=init,
                         frames=n_frames, interval=50, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

interact(conical_pendulum,
         omega=FloatSlider(min=2.0, max=8.0, step=0.2, value=3.0, description='omega (rad/s)'),
         L=FloatSlider(min=0.5, max=3.0, step=0.1, value=1.5, description='L (m)'));

---
## 10. Worked Examples

### Example 1 -- Car on a flat curve

A 1200 kg car rounds a flat curve of radius 80 m at 25 m/s. What friction force is required?

$$F_f = \frac{mv^2}{r} = \frac{1200 \times 25^2}{80} = 9375 \text{ N}$$

In [ ]:
m, v, r = 1200, 25, 80
F_c = m * v**2 / r
print(f'Required friction force: {F_c:.0f} N')
print(f'Required mu_s >= F_c / (mg) = {F_c / (m*9.81):.3f}')

### Example 2 -- Banked road

A highway curve has radius 120 m and is banked at 15 degrees. What is the ideal (no-friction) speed?

In [ ]:
r, theta_deg = 120, 15
g = 9.81
theta = np.radians(theta_deg)
v_ideal = np.sqrt(r * g * np.tan(theta))
print(f'Ideal speed: {v_ideal:.2f} m/s = {v_ideal*3.6:.1f} km/h')

### Example 3 -- Loop-the-loop

A roller coaster loop has radius 8 m. What minimum speed is needed at the bottom to complete the loop?

In [ ]:
R = 8.0
g = 9.81
v_top_min = np.sqrt(g * R)
v_bot_min = np.sqrt(5 * g * R)
print(f'Min speed at top: {v_top_min:.2f} m/s')
print(f'Min speed at bottom: {v_bot_min:.2f} m/s = {v_bot_min*3.6:.1f} km/h')

---
## Problem Set

**Core Mastery Workflow — For each problem: Draw the diagram -> Identify the principle -> Write the equation -> Predict -> Verify.**

**Instructions:** Solve each problem in the code cell provided. Show your work using Python calculations. Use `numpy` functions where appropriate. Problems are graded by difficulty:
- **L1 (Basic):** Single-concept, straightforward calculation
- **L2 (Intermediate):** Multi-step, combines two or more concepts
- **L3 (Challenge):** Multi-concept integration, deeper analysis required

### L1 -- P1: Centripetal Acceleration

A car travels around a circular track of radius 50.0 m at a constant speed of 15.0 m/s. (a) What is the centripetal acceleration? (b) If the car's mass is 1200 kg, what centripetal force is required?

<details><summary>Answer</summary>(a) a_c = v²/r = 225/50 = 4.50 m/s². (b) F_c = ma_c = 1200(4.50) = 5400 N.</details>

In [ ]:
# ✏️ [P1] Your solution here


### L1 -- P2: Period and Frequency

A centrifuge rotor spins at 3000 rpm with a radius of 0.15 m. (a) What is the frequency in Hz? (b) What is the period? (c) What is the centripetal acceleration at the rim, expressed in multiples of g?

<details><summary>Answer</summary>(a) f = 3000/60 = 50 Hz. (b) T = 1/f = 0.020 s. (c) ω = 2πf = 314.2 rad/s; a_c = ω²r = (314.2)²(0.15) = 14,808 m/s² = 1510 g.</details>

In [ ]:
# ✏️ [P2] Your solution here


### L1 -- P3: Flat Curve Friction

A 1500 kg car rounds a flat curve of radius 80.0 m. The coefficient of static friction between the tires and road is 0.55. What is the maximum speed the car can have without sliding?

<details><summary>Answer</summary>μ_s mg = mv²/r → v = √(μ_s g r) = √(0.55 × 9.81 × 80) = 20.8 m/s = 74.8 km/h.</details>

In [ ]:
# ✏️ [P3] Your solution here


### L1 -- P4: Satellite Orbital Speed

A satellite orbits Earth at an altitude of 400 km above the surface. Earth's radius is 6371 km and g at that altitude is approximately 8.69 m/s². Find the orbital speed and period.

<details><summary>Answer</summary>r = 6371 + 400 = 6771 km = 6.771 × 10⁶ m. v = √(g·r) = √(8.69 × 6.771×10⁶) = 7670 m/s = 27,600 km/h. T = 2πr/v = 2π(6.771×10⁶)/7670 = 5545 s = 92.4 min.</details>

In [ ]:
# ✏️ [P4] Your solution here


### L2 -- P5: Banked Curve Design

A highway curve of radius 200 m is designed for traffic at 90 km/h. (a) What bank angle is needed with no friction? (b) If the road is wet (μ_s = 0.20), what are the minimum and maximum safe speeds? (c) Express the maximum speed in km/h.

<details><summary>Answer</summary>(a) v = 90/3.6 = 25 m/s; tanθ = v²/(rg) = 625/(200×9.81) = 0.3185; θ = 17.7°. (b) v_min = √[rg(tanθ − μ)/(1 + μ tanθ)] = √[200(9.81)(0.3185−0.20)/(1+0.0637)] = √[200(9.81)(0.1185)/1.0637] = 14.8 m/s. v_max = √[rg(tanθ + μ)/(1 − μ tanθ)] = √[200(9.81)(0.5185)/0.9363] = 32.9 m/s. (c) v_max = 32.9 × 3.6 = 118.5 km/h.</details>

In [ ]:
# ✏️ [P5] Your solution here


### L2 -- P6: Vertical Loop -- Normal Force

A 0.50 kg ball on a string moves in a vertical circle of radius 1.20 m. At the top of the circle, the speed is 4.0 m/s. (a) What is the tension in the string at the top? (b) Using energy conservation, find the speed at the bottom. (c) What is the tension at the bottom?

<details><summary>Answer</summary>(a) At top: T + mg = mv²/r → T = m(v²/r − g) = 0.50(16/1.20 − 9.81) = 0.50(13.33 − 9.81) = 1.76 N. (b) ½mv_top² + mg(2r) = ½mv_bot² → v_bot = √(v_top² + 4gr) = √(16 + 47.09) = 7.94 m/s. (c) T − mg = mv_bot²/r → T = m(v_bot²/r + g) = 0.50(63.04/1.20 + 9.81) = 0.50(52.53 + 9.81) = 31.2 N.</details>

In [ ]:
# ✏️ [P6] Your solution here


### L2 -- P7: Conical Pendulum

A 0.30 kg mass hangs from a 0.80 m string and swings in a horizontal circle, making an angle of 25° with the vertical. (a) Find the radius of the circular path. (b) Find the speed of the mass. (c) Find the tension in the string. (d) Find the period of revolution.

<details><summary>Answer</summary>(a) r = L sinθ = 0.80 sin25° = 0.338 m. (b) tanθ = v²/(rg) → v = √(rg tanθ) = √(0.338 × 9.81 × 0.466) = 1.25 m/s. (c) T cosθ = mg → T = mg/cos25° = 0.30(9.81)/0.906 = 3.25 N. (d) T_period = 2πr/v = 2π(0.338)/1.25 = 1.70 s.</details>

In [ ]:
# ✏️ [P7] Your solution here


### L2 -- P8: Car Over a Hill

A 1000 kg car drives over the top of a hill that can be approximated as a circular arc of radius 40.0 m. (a) At what speed does the car begin to lose contact with the road (normal force = 0)? (b) If the car goes over the hill at 15.0 m/s, what is the normal force on the car? (c) What does the driver feel in terms of apparent weight?

<details><summary>Answer</summary>(a) mg = mv²/r → v = √(gr) = √(9.81 × 40) = 19.8 m/s. (b) mg − N = mv²/r → N = m(g − v²/r) = 1000(9.81 − 225/40) = 1000(9.81 − 5.625) = 4185 N. (c) Apparent weight = N = 4185 N, which is 4185/9810 = 42.7% of true weight. The driver feels lighter.</details>

In [ ]:
# ✏️ [P8] Your solution here


### L3 -- P9: Loop-the-Loop with Friction

A small block slides from rest down a frictionless ramp of height h and enters a vertical loop of radius R = 1.50 m. The loop has a coefficient of kinetic friction μ_k = 0.10. (a) Find the minimum height h such that the block barely maintains contact at the top of the loop. (Hint: at the top, use both Newton's 2nd law and energy conservation with friction loss.) (b) Compare this to the frictionless case.

<details><summary>Answer</summary>(a) At top of loop: mg = mv_top²/R → v_top² = gR. Energy: mgh = ½mv_top² + mg(2R) + W_friction. The friction work on the loop is approximately μ_k mg × (πR + πR) = μ_k mg(2πR) (rough estimate for entire loop, N varies). Simplified: mgh = ½m(gR) + 2mgR + μ_k mg(2πR). h = R/2 + 2R + 2πμ_k R = R(2.5 + 2π(0.10)) = 1.50(2.5 + 0.628) = 4.69 m. (b) Frictionless: h = 2.5R = 3.75 m. Friction requires 25% more height.</details>

In [ ]:
# ✏️ [P9] Your solution here


### L3 -- P10: Banked Curve Design for an Autonomous Vehicle

You are designing a banked test track for autonomous vehicles. The curve has radius 150 m. Vehicles will travel between 40 km/h and 120 km/h. The tire-road friction coefficient is μ_s = 0.40. (a) Find the bank angle that requires no friction at 80 km/h (the median speed). (b) Verify that vehicles at 40 km/h and 120 km/h can navigate the curve without sliding. (c) What is the absolute maximum speed before sliding occurs? (d) At maximum speed, what is the centripetal acceleration in g's?

<details><summary>Answer</summary>(a) v = 80/3.6 = 22.22 m/s; tanθ = v²/(rg) = 493.8/1471.5 = 0.3356; θ = 18.5°. (b) v_min = √[rg(tanθ − μ)/(1+μtanθ)] = √[150(9.81)(−0.0644)/1.1342] -- negative under sqrt means friction can hold even at v=0, so 40 km/h (11.1 m/s) is fine. v_max = √[150(9.81)(0.7356)/0.8658] = √[150(9.81)(0.8497)] = 35.4 m/s = 127.4 km/h > 120 km/h, so safe. (c) v_max = 35.4 m/s = 127.4 km/h. (d) a_c = v²/r = 1253/150 = 8.35 m/s² = 0.85 g.</details>

In [ ]:
# ✏️ [P10] Your solution here


---
## 11. Bridge to Next Week

This week we saw that circular motion requires a net inward force. Next week we leave forces behind temporarily and adopt an **energy** viewpoint:

- **Work** done by a force along a displacement
- **Kinetic energy** and the **work-energy theorem**
- **Potential energy** (gravitational and elastic)
- **Conservation of mechanical energy**

The energy approach often makes problems *much* simpler -- for example, the loop-the-loop minimum speed we derived today comes directly from energy conservation, which we will explore in depth next week.